#### 0. Configuración Inicial
Importar las librerías necesarias y cargar el dataset de actividad.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cargar el dataset
df = pd.read_csv('docs/product_activity.csv')

# Guardar el total original antes de cualquier limpieza
FILAS_ORIGINALES = len(df)

#### 1. Exploración Inicial
Medir antes de limpiar: entender la estructura del dataset, detectar nulos, duplicados y valores inconsistentes.

##### 1.1 — Vista general del dataset
Revisar las primeras filas, tipos de datos y estadísticas descriptivas para entender la forma y distribución del dataset.

In [ ]:
print("Head")
display(df.head())
print("\nInfo")
df.info()
print("\nDescribe")
display(df.describe())

##### 1.2 — Conteo de nulos y duplicados
Identificar cuántos valores faltantes hay por columna y cuántas filas duplicadas exactas existen.

In [ ]:
# Contar nulos por columna
print("Nulos por columna:")
display(df.isnull().sum())

# Contar filas duplicadas exactas
print(f"\nFilas duplicadas exactas: {df.duplicated().sum()}")

##### 1.3 — Valores únicos y frecuencias
Revisar qué valores existen en las columnas categóricas para detectar variantes, typos y valores fuera de diccionario.

In [ ]:
# Frecuencias de cada columna categórica
print("plan_type:")
display(df["plan_type"].value_counts())

print("\npost_category:")
display(df["post_category"].value_counts())

print("\ndevice_type:")
display(df["device_type"].value_counts())

##### 1.4 — Chequeos lógicos
Verificar la coherencia temporal: ¿hay posts creados antes del signup? ¿La columna `days_since_signup` es consistente con las fechas?

In [ ]:
# Convertir fechas temporalmente para el chequeo
signup = pd.to_datetime(df["created_at"], errors="coerce")
post = pd.to_datetime(df["post_created_at"], errors="coerce")

# ¿Cuántos posts ocurren antes del signup?
posts_before_signup = (post < signup).sum()
print(f"Posts antes del signup: {posts_before_signup}")

# ¿days_since_signup es consistente?
dias_calculados = (post - signup).dt.days
dias_originales = df["days_since_signup"]
inconsistentes = (dias_calculados != dias_originales).sum()
print(f"Days_since_signup inconsistentes: {inconsistentes}")

#### 2. Limpieza Básica con Criterio
Aplicar limpieza con diccionarios fijos, corregir fechas y separar filas con errores duros.

##### 2.1 — Eliminar duplicados exactos

In [ ]:
# Registrar la cantidad de duplicados antes de eliminar
duplicados_removidos = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicados removidos: {duplicados_removidos}")
print(f"Filas restantes: {len(df)}")

##### 2.2 — Normalización canónica
Mapear variantes (mayúsculas, espacios, typos) a diccionarios fijos para `plan_type`, `device_type` y `post_category`.

In [ ]:
# Paso previo: poner todo en minúsculas y quitar espacios
df["plan_type"] = df["plan_type"].str.strip().str.lower()
df["device_type"] = df["device_type"].str.strip().str.lower()
df["post_category"] = df["post_category"].str.strip().str.lower()

# Definir diccionarios de mapeo para variantes conocidas
mapeo_plan = {
    "free": "free",
    "pro": "pro",
    "enterprise": "enterprise"
}

mapeo_device = {
    "web": "web",
    "mobile": "mobile",
    "desktop": "desktop"
}

mapeo_category = {
    "sport": "sports",
    "sporst": "sports",
    "sp0rts": "sports",
    "tech": "tech",
    "tehc": "tech",
    "life": "life",
    "lfe": "life",
    "gaming": "gaming",
    "gamming": "gaming",
    "music": "music",
    "musc": "music",
    "education": "education",
    "educatoin": "education",
    "health": "health",
    "healt": "health",
    "science": "science",
    "sciense": "science",
    "travel": "travel",
    "trvael": "travel",
    "finance": "finance",
    "finanse": "finance"
}

df["plan_type"] = df["plan_type"].replace(mapeo_plan)
df["device_type"] = df["device_type"].replace(mapeo_device)
df["post_category"] = df["post_category"].replace(mapeo_category)

# Verificar resultado de la normalización
print("plan_type después de normalizar:")
display(df["plan_type"].value_counts())

print("\ndevice_type después de normalizar:")
display(df["device_type"].value_counts())

print("\npost_category después de normalizar:")
display(df["post_category"].value_counts())

##### 2.3 — Convertir fechas a datetime
Convertir las columnas de fecha y reportar cuántas no se pudieron parsear.

In [ ]:
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df["post_created_at"] = pd.to_datetime(df["post_created_at"], errors="coerce")

# Reportar fechas no parseables
fechas_nulas_signup = df["created_at"].isna().sum()
fechas_nulas_post = df["post_created_at"].isna().sum()
print(f"Fechas no parseables - signup: {fechas_nulas_signup}")
print(f"Fechas no parseables - post: {fechas_nulas_post}")

##### 2.4 — Recálculo obligatorio de days_since_signup
Crear `days_since_signup_calc` a partir de las fechas y comparar con la columna original.

In [ ]:
# Crear la columna recalculada
df["days_since_signup_calc"] = (df["post_created_at"] - df["created_at"]).dt.days

# Comparar con el original
mismatches = (df["days_since_signup"] != df["days_since_signup_calc"]).sum()
print(f"Mismatches en days_since_signup: {mismatches}")

##### 2.5 — Quarantine (Cuarentena)
Separar filas con errores duros (post < signup, fechas no parseables, valores fuera de diccionario) en un DataFrame aparte con `reason_code`.

In [ ]:
# Identificar filas con errores duros y asignar motivo

# Condición 1: post_created_at < created_at (post antes del signup)
cond_fecha = df["post_created_at"] < df["created_at"]

# Condición 2: fechas no parseables (NaT)
cond_nat = df["created_at"].isna() | df["post_created_at"].isna()

# Condición 3: valores que no están en el diccionario fijo
planes_validos = ["free", "pro", "enterprise"]
devices_validos = ["web", "mobile", "desktop"]
categorias_validas = ["tech", "life", "sports", "science", "finance", "gaming", "music", "health", "education", "travel"]

cond_plan = ~df["plan_type"].isin(planes_validos)
cond_device = ~df["device_type"].isin(devices_validos)
cond_cat = ~df["post_category"].isin(categorias_validas)

# Crear columna reason_code
df["reason_code"] = ""
df.loc[cond_fecha, "reason_code"] += "post_antes_signup;"
df.loc[cond_nat, "reason_code"] += "fechas_no_parseables;"
df.loc[cond_plan, "reason_code"] += "plan_invalido;"
df.loc[cond_device, "reason_code"] += "device_invalido;"
df.loc[cond_cat, "reason_code"] += "categoria_invalida;"

# Separar: cuarentena vs core
mascara_quarantine = df["reason_code"] != ""
df_quarantine = df[mascara_quarantine].copy()
df_core = df[~mascara_quarantine].copy()

# Limpiar columna temporal
df_core = df_core.drop(columns=["reason_code"])

print(f"Filas en cuarentena: {len(df_quarantine)}")
print(f"Filas en core (limpio): {len(df_core)}")

#### 3. Data Quality Report
Resumen de integridad: filas RAW vs CORE, porcentaje de cuarentena, duplicados removidos y mismatches en fechas.

In [ ]:
filas_raw = FILAS_ORIGINALES
filas_core = len(df_core)
filas_quarantine = len(df_quarantine)
pct_quarantine = (filas_quarantine / filas_raw) * 100
pct_mismatches = (mismatches / filas_raw) * 100

dqr = pd.DataFrame({
    'Métrica': ['Filas RAW', 'Filas CORE', 'Filas Quarantine', '% Quarantine', 'Duplicados Removidos', '% Mismatches Fechas'],
    'Valor': [filas_raw, filas_core, filas_quarantine, f"{pct_quarantine:.2f}%", duplicados_removidos, f"{pct_mismatches:.2f}%"]
})
display(dqr)

#### 4. Métricas y Análisis
Distribuciones de volumen, engagement por segmento y comparación evento vs usuario.

##### 4.1 — Distribuciones (Volumen)
Usuarios únicos por plan y actividad (cantidad de posts) por país, categoría y dispositivo.

In [ ]:
# Usuarios únicos por plan
print("Usuarios únicos por plan:")
display(df_core.groupby("plan_type")["user_id"].nunique())

# Actividad (#posts) por país
print("\nActividad por país:")
display(df_core["country"].value_counts())

# Actividad (#posts) por categoría
print("\nActividad por categoría:")
display(df_core["post_category"].value_counts())

# Actividad (#posts) por dispositivo
print("\nActividad por dispositivo:")
display(df_core["device_type"].value_counts())

# Gráfico de barras: posts por plan
df_core["plan_type"].value_counts().plot(kind="bar", title="Posts por Plan Type")
plt.show()

##### 4.2 — Engagement (Votos)
Votos recibidos por plan, país, categoría y dispositivo (media y mediana).

In [ ]:
# Votos por plan: media y mediana
print("Votos por plan (media y mediana):")
display(df_core.groupby("plan_type")["votes_received"].agg(["mean", "median"]))

# Votos por país
print("\nVotos por país (media y mediana):")
display(df_core.groupby("country")["votes_received"].agg(["mean", "median"]))

# Votos por categoría
print("\nVotos por categoría (media y mediana):")
display(df_core.groupby("post_category")["votes_received"].agg(["mean", "median"]))

# Votos por dispositivo
print("\nVotos por dispositivo (media y mediana):")
display(df_core.groupby("device_type")["votes_received"].agg(["mean", "median"]))

##### 4.3 — Promedios e Interpretación

**Interpretación:** La unidad de análisis en este dataset es el *evento* (cada fila es un post). Esto significa que un usuario con muchísimos posts impacta fuertemente en el promedio general (sesgo de heavy users o outliers), ya que sus votos se registran repetidamente, distorsionando la realidad del usuario promedio.

In [ ]:
# Promedio de votos por plan
votos_plan = df_core.groupby("plan_type")["votes_received"].mean()
print("Promedio de votos por plan:")
display(votos_plan)

# Posts promedio por usuario
posts_promedio_por_usuario = df_core.groupby("user_id")["post_id"].count().mean()
print(f"\nPosts promedio por usuario: {posts_promedio_por_usuario:.2f}")

##### 4.4 — Evento vs Usuario

**Por qué difieren:** En el nivel *evento*, los usuarios más activos sobrerrepresentan la muestra (si un usuario hace 100 posts, su media pesará 100 veces más). Al calcular por *usuario*, a cada persona se le asigna el mismo peso (1 usuario = 1 promedio), limpiando el efecto que provocan los usuarios extremos.

In [ ]:
# Promedio de votos por FILA (nivel evento)
promedio_evento = df_core["votes_received"].mean()
print(f"Promedio de votos por evento (fila): {promedio_evento:.4f}")

# Promedio de votos agrupado por USUARIO
promedio_usuario = df_core.groupby("user_id")["votes_received"].mean().mean()
print(f"Promedio de votos por usuario: {promedio_usuario:.4f}")

#### 5. Concentración y Temporalidad
Analizar cómo se distribuye la actividad entre usuarios y cómo evoluciona en el tiempo.

##### 5.1 — Concentración (Top 1%)
¿Qué porcentaje de posts y votos viene del top 1% de usuarios?

In [ ]:
# Posts y votos por usuario
posts_por_usuario = df_core.groupby("user_id")["post_id"].count()
votos_por_usuario = df_core.groupby("user_id")["votes_received"].sum()

# Calcular el umbral del top 1%
umbral_posts = posts_por_usuario.quantile(0.99)
umbral_votos = votos_por_usuario.quantile(0.99)

# Filtrar top 1%
top_posts = posts_por_usuario[posts_por_usuario >= umbral_posts]
top_votos = votos_por_usuario[votos_por_usuario >= umbral_votos]
print(f"Top 1% usuarios por posts: {len(top_posts)}")

# Calcular porcentaje del total
pct_top_posts = (top_posts.sum() / posts_por_usuario.sum()) * 100
pct_top_votos = (top_votos.sum() / votos_por_usuario.sum()) * 100
print(f"Top 1% usuarios representan {pct_top_posts:.2f}% de los posts")
print(f"Top 1% usuarios representan {pct_top_votos:.2f}% de los votos")

##### 5.2 — Tendencia temporal
Actividad y engagement por mes para detectar patrones temporales.

In [ ]:
# Crear columna de mes
df_core["month"] = df_core["post_created_at"].dt.to_period("M")

# Actividad por mes
actividad_mes = df_core.groupby("month")["post_id"].count()
actividad_mes.plot(kind="line", title="Actividad por Mes")
plt.show()

# Engagement por mes (votos promedio por post)
engagement_mes = df_core.groupby("month")["votes_received"].mean()
engagement_mes.plot(kind="line", title="Engagement por Mes")
plt.show()

#### 6. Product Decisions
Decisiones basadas en evidencia a partir del análisis realizado.

##### Preguntas

- **¿Qué segmento priorizarías y por qué?**
El segmento Enterprise/Pro, ya que muestran en la distribución métricas consistentemente valiosas y retienen más actividad sostenida. Además, enfoques de retención hacia usuarios de mobile suelen ser clave.

- **¿Qué parte del tablero "mentía" antes de limpiar?**
La métrica `days_since_signup`. Mostraba un porcentaje importante de mismatches comparado con un recálculo limpio, generando posibles errores en el tiempo de vida (LTV) del usuario.

- **¿Qué nuevo dato agregarías al tracking?**
Tiempo promedio leyendo el post o si incluye multimedia, para que el número de votos no sea el único reflejo de valor real (engagement puro).

##### Acciones

1. **Campaña de Retención:** El Top 1% agrupa demasiados posts. Se debe incentivar al resto de la base a publicar, por ejemplo, dando un "logro/bonus" en el primer mes de uso.

2. **Mejora Mobile:** Si la categoría o equipo móvil domina, se debe priorizar recursos de ingeniería en optimizar su interfaz en próximas versiones.

##### Limitaciones del dataset

- No hay datos de retención ni duración de sesión, lo que limita la evaluación de engagement real.
- `user_total_posts` es un atributo repetido por fila; puede sesgar promedios si no se agrupa por usuario.
- No hay información sobre la calidad del contenido del post (longitud, multimedia, etc.).
- El período temporal del dataset puede no ser representativo de tendencias a largo plazo.

#### 7. Exportación de Archivos

In [ ]:
# 1. Dataset limpio
df_core.to_csv("clean_product_activity.csv", index=False)

# 2. Dataset cuarentena
df_quarantine.to_csv("quarantine_product_activity.csv", index=False)

# 3. Tabla resumen de métricas clave
metricas = pd.DataFrame({
    "Métrica": ["usuarios_unicos", "total_posts_core", "promedio_votos_evento", "promedio_votos_usuario", "pct_top_1_posts", "pct_top_1_votos"],
    "Valor": [df_core["user_id"].nunique(), len(df_core), promedio_evento, promedio_usuario, pct_top_posts, pct_top_votos]
})
metricas.to_csv("metrics_summary.csv", index=False)

print("Archivos exportados correctamente:")
print("  - clean_product_activity.csv")
print("  - quarantine_product_activity.csv")
print("  - metrics_summary.csv")